##### Prompt Chain Workflow

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from IPython.display import display, Image


In [18]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    rating: int

In [4]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [5]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    state['outline'] = outline
    return state

In [12]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Create a blog on the topic - {title} according to the given outline - {outline}'
    blog = model.invoke(prompt).content
    state['content'] = blog
    
    return state

In [19]:
def rate_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    blog = state['content']

    prompt = f'Based on the title - {title} and outline - {outline}, rate this blog - {blog} on a scale between 1-10'
    score = model.invoke(prompt).content
    state['rating'] = score
    
    return state


In [23]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('rate_blog', rate_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'rate_blog')
graph.add_edge('rate_blog', END)

workflow = graph.compile()

In [24]:
initial_state = {'title': 'Black hole'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'title': 'Black hole', 'outline': '## Blog Post Outline: Unveiling the Mysteries of Black Holes\n\n**Topic:** Black Holes: The Universe\'s Ultimate Enigmas\n\n**Target Audience:** General audience with an interest in space and science, curious learners, and budding astrophysicists.\n\n**Goal:** To provide a comprehensive yet accessible overview of black holes, demystifying their nature, formation, and the profound implications they hold for our understanding of the universe.\n\n---\n\n**I. Introduction: The Allure of the Void**\n\n    *   **A. Hook:** Start with a captivating image or a thought-provoking question about black holes (e.g., "What happens when gravity becomes so strong that not even light can escape?").\n    *   **B. Brief Definition:** Introduce black holes as regions of spacetime with extreme gravity.\n    *   **C. Historical Context (briefly):** Mention early theoretical ideas (e.g., John Michell, Pierre-Simon Laplace) and Einstein\'s General Relativity as the foundati

In [25]:
print(final_state['rating'])

This is an excellent and comprehensive blog post outline. Here's a rating and breakdown:

**Overall Rating: 9.5/10**

This outline is incredibly well-structured, thorough, and engaging. It covers all the essential aspects of black holes in a logical flow, making it perfect for a general audience while still offering depth for those with more scientific curiosity.

Here's a breakdown of its strengths:

**Strengths:**

*   **Clear and Engaging Introduction (I):** The hook is strong, the historical context is well-placed, and the roadmap sets clear expectations.
*   **Logical Progression of Concepts (II-VIII):** The outline moves from basic definitions and anatomy to formation, types, effects, detection, broader implications, and finally, future research. This structured approach makes complex topics digestible.
*   **Comprehensive Coverage:** It touches upon all the key aspects of black holes that a general audience would be interested in, including the "what," "how," "types," "effects,"